# IMDb Movie Review Sentiment Analysis with PyTorch

This notebook documents an educational end-to-end sentiment-analysis pipeline using an **Embedding + LSTM** classifier.

**Goal:** classify movie reviews as negative (`0`) or positive (`1`).

The notebook keeps a few useful learning experiments—tokenization, sequence padding, tensor shapes, and the LSTM architecture—while avoiding repeated dataset loads and duplicate training loops.

> Run the notebook from the project environment. The project modules are imported from `src/`, so start Jupyter from the repository root or keep the path setup cell below.

## 1. Environment and imports

We use Hugging Face `datasets` to load IMDb, NLTK for tokenization, PyTorch for the model, and scikit-learn for evaluation.

In [ ]:
from pathlib import Path
import sys
import json
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Support running this notebook either from the project root or from notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Could not find the project 'src' folder. Run Jupyter from the project root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocess import clean_text, tokenize, encode, pad_or_truncate
from src.model import SentimentLSTM

print("Project root:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Load and inspect the IMDb dataset

The standard split contains 25,000 training reviews and 25,000 test reviews. We build the vocabulary using **training reviews only** to avoid using test data during vocabulary construction.

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")
train_data = dataset["train"]
test_data = dataset["test"]

print(dataset)
print(f"Training reviews: {len(train_data):,}")
print(f"Test reviews: {len(test_data):,}")

sample = train_data[0]
print("\nExample label:", sample["label"], "(0 = negative, 1 = positive)")
print(sample["text"][:700])

In [ ]:
train_label_counts = Counter(train_data["label"])
test_label_counts = Counter(test_data["label"])

print("Training label counts:", train_label_counts)
print("Test label counts:", test_label_counts)

fig, ax = plt.subplots(figsize=(6, 4))
labels = ["Negative", "Positive"]
counts = [train_label_counts[0], train_label_counts[1]]
ax.bar(labels, counts)
ax.set_title("Training label distribution")
ax.set_ylabel("Number of reviews")
plt.show()

## 3. Text cleaning and tokenization

The project's preprocessing functions remove HTML tags, normalize whitespace, lowercase the text, and tokenize it. We use the same functions for vocabulary creation, training, and later predictions.

In [ ]:
example_text = "This movie was <b>REALLY good</b>!<br />I loved it."

print("Original:")
print(example_text)
print("\nCleaned:")
print(clean_text(example_text))
print("\nTokens:")
print(tokenize(example_text))

In [ ]:
sample_review = train_data[0]["text"]
sample_tokens = tokenize(sample_review)

print("Label:", train_data[0]["label"])
print("Token count:", len(sample_tokens))
print("First 30 tokens:", sample_tokens[:30])

## 4. Build the vocabulary

The vocabulary is based only on the training split. The two special tokens are:

- `<PAD>` (`0`): fills sequences to a common length.
- `<UNK>` (`1`): represents tokens that are not in the vocabulary.

We keep the 29,998 most frequent training tokens in addition to these two special tokens, for a maximum vocabulary size of 30,000.

In [ ]:
MAX_VOCAB_SIZE = 30_000
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

token_counter = Counter()
for review in train_data:
    token_counter.update(tokenize(review["text"]))

print("Unique training tokens:", len(token_counter))
print("Total token occurrences:", token_counter.total())
print("\nMost common 15 tokens:")
for token, count in token_counter.most_common(15):
    print(f"{token!r}: {count:,}")

In [ ]:
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}

for token, _count in token_counter.most_common(MAX_VOCAB_SIZE - len(vocab)):
    if token not in vocab:
        vocab[token] = len(vocab)

print("Vocabulary size:", len(vocab))
print("PAD ID:", vocab[PAD_TOKEN])
print("UNK ID:", vocab[UNK_TOKEN])
print("ID for 'movie':", vocab.get("movie"))

### Learning check: encode one review

Each token is replaced by an integer ID. Unknown tokens use the `<UNK>` ID.

In [ ]:
encoded_review = encode(sample_review, vocab)
print("Number of tokens:", len(sample_tokens))
print("Number of token IDs:", len(encoded_review))
print("First 30 IDs:", encoded_review[:30])

## 5. Choose a sequence length and pad/truncate

Neural-network batches need tensors of consistent shape. We keep the first 300 token IDs from longer reviews and append the padding ID to shorter reviews.

This is a practical fixed-length choice, not a claim that every review contains exactly 300 meaningful tokens.

In [ ]:
MAX_LENGTH = 300

short_example = [12, 25, 8]
long_example = list(range(400))

padded_example = pad_or_truncate(short_example, max_length=MAX_LENGTH, pad_token_id=vocab[PAD_TOKEN])
truncated_example = pad_or_truncate(long_example, max_length=MAX_LENGTH, pad_token_id=vocab[PAD_TOKEN])

print("Short sequence length:", len(padded_example))
print("Short sequence after padding:", padded_example[:10])
print("Long sequence length after truncation:", len(truncated_example))
print("Last 5 retained IDs:", truncated_example[-5:])

In [ ]:
# Inspect the token-length distribution on a small sample before full preprocessing.
sample_lengths = [len(tokenize(train_data[i]["text"])) for i in range(min(1000, len(train_data)))]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sample_lengths, bins=40)
ax.axvline(MAX_LENGTH, linestyle="--", label=f"MAX_LENGTH = {MAX_LENGTH}")
ax.set_title("Review token lengths (first 1,000 training reviews)")
ax.set_xlabel("Number of tokens")
ax.set_ylabel("Number of reviews")
ax.legend()
plt.show()

print("Median length in this sample:", int(np.median(sample_lengths)))
print("95th percentile in this sample:", int(np.percentile(sample_lengths, 95)))

## 6. Create PyTorch datasets and data loaders

The dataset below tokenizes and pads each review when it is requested. This is straightforward and easy to read, although it repeats preprocessing across epochs. For this project, the notebook uses a cached dataset implementation instead, so each review is encoded once before training.

In [ ]:
class EncodedIMDBDataset(Dataset):
    def __init__(self, data, vocab, max_length=300):
        self.reviews = []
        self.labels = []

        for item in data:
            token_ids = encode(item["text"], vocab)
            token_ids = pad_or_truncate(
                token_ids,
                max_length=max_length,
                pad_token_id=vocab[PAD_TOKEN],
            )
            self.reviews.append(token_ids)
            self.labels.append(int(item["label"]))

        self.reviews = torch.tensor(self.reviews, dtype=torch.long)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.reviews[index], self.labels[index]

In [ ]:
BATCH_SIZE = 32

train_dataset = EncodedIMDBDataset(train_data, vocab, max_length=MAX_LENGTH)
test_dataset = EncodedIMDBDataset(test_data, vocab, max_length=MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train tensor shape:", train_dataset.reviews.shape)
print("Test tensor shape:", test_dataset.reviews.shape)
print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

### Learning check: inspect a batch

A batch of 32 reviews becomes a tensor with shape `(32, 300)`. Labels have shape `(32,)`.

In [ ]:
reviews, labels = next(iter(train_loader))
print("Reviews shape:", reviews.shape)
print("Labels shape:", labels.shape)
print("First 10 IDs in first review:", reviews[0, :10].tolist())
print("First label:", labels[0].item())

## 7. Understand the model components

Before using the complete model, inspect the shapes produced by an embedding layer, an LSTM, and a classifier. These are small, untrained components used only to illustrate tensor dimensions.

In [ ]:
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 100
HIDDEN_SIZE = 64
NUM_CLASSES = 2

demo_embedding = nn.Embedding(
    num_embeddings=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    padding_idx=vocab[PAD_TOKEN],
)

demo_batch = reviews[:4]
embedded_batch = demo_embedding(demo_batch)
print("Token IDs:", demo_batch.shape)            # (batch, sequence length)
print("Embeddings:", embedded_batch.shape)       # (batch, sequence length, embedding dimension)

In [ ]:
demo_lstm = nn.LSTM(
    input_size=EMBEDDING_DIM,
    hidden_size=HIDDEN_SIZE,
    batch_first=True,
)

lstm_output, (hidden_state, cell_state) = demo_lstm(embedded_batch)
print("LSTM output:", lstm_output.shape)
print("Hidden state:", hidden_state.shape)
print("Cell state:", cell_state.shape)

demo_classifier = nn.Linear(HIDDEN_SIZE, NUM_CLASSES)
demo_logits = demo_classifier(hidden_state[-1])
print("Class logits:", demo_logits.shape)

## 8. Create the complete SentimentLSTM model

The project implementation in `src/model.py` combines the embedding layer, LSTM, and final linear classifier. It uses `pack_padded_sequence` so that the LSTM processes each review's actual non-padding length rather than treating trailing padding as review content.

The model returns **logits** (two raw class scores). `CrossEntropyLoss` expects logits, so we do not apply softmax during training.

In [ ]:
model = SentimentLSTM(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_size=HIDDEN_SIZE,
    num_classes=NUM_CLASSES,
    padding_idx=vocab[PAD_TOKEN],
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

### Learning check: one forward pass

This is only a shape check; it does not train the model.

In [ ]:
model.eval()
with torch.no_grad():
    sample_logits = model(reviews[:4].to(device))

print("Logits shape:", sample_logits.shape)  # (batch size, 2)

## 9. Train the model

Each training batch follows the standard PyTorch loop:

1. Clear gradients.
2. Run a forward pass.
3. Calculate the loss.
4. Backpropagate the loss.
5. Update model parameters.

Training for five epochs can take a while depending on the computer. Results can differ between runs because of random initialization, data shuffling, hardware, and software versions.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for reviews, labels in loader:
        reviews = reviews.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(reviews)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        predictions = logits.argmax(dim=1)
        correct_predictions += (predictions == labels).sum().item()
        total_samples += batch_size

    return total_loss / total_samples, correct_predictions / total_samples

In [ ]:
EPOCHS = 5
history = {"loss": [], "accuracy": []}

for epoch in range(EPOCHS):
    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    history["loss"].append(train_loss)
    history["accuracy"].append(train_accuracy)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Accuracy: {train_accuracy * 100:.2f}%"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, EPOCHS + 1), history["loss"], marker="o")
ax.set_title("Training loss by epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.set_xticks(range(1, EPOCHS + 1))
plt.show()

## 10. Evaluate on the held-out test set

The test set is used only for evaluation—not for building the vocabulary or updating model weights.

In [ ]:
def evaluate_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for reviews, labels in loader:
            reviews = reviews.to(device)
            labels = labels.to(device)

            logits = model(reviews)
            loss = criterion(logits, labels)
            predictions = logits.argmax(dim=1)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            correct_predictions += (predictions == labels).sum().item()
            total_samples += batch_size
            all_predictions.extend(predictions.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    return (
        total_loss / total_samples,
        correct_predictions / total_samples,
        all_labels,
        all_predictions,
    )

test_loss, test_accuracy, all_labels, all_predictions = evaluate_model(
    model, test_loader, criterion, device
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["Negative", "Positive"],
        digits=4,
    )
)

In [ ]:
cm = confusion_matrix(all_labels, all_predictions)
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"],
)
display.plot(values_format="d")
plt.title("IMDb test-set confusion matrix")
plt.show()

print("Rows = actual labels; columns = predicted labels")
print(cm)

### Reference result from the earlier completed run

The previously saved model evaluation reported **80.24% test accuracy** and **0.4461 test loss**. A fresh training run in this notebook may produce different values. Use the metrics printed by the current run as the authoritative results for that run.

## 11. Try predictions on new reviews

The prediction function uses the same vocabulary, tokenization, and sequence length as training. The displayed softmax score is the model's score for its predicted class; it should not automatically be interpreted as a calibrated real-world probability.

In [ ]:
def predict_sentiment(text, model, vocab, device, max_length=300):
    token_ids = encode(text, vocab)
    token_ids = pad_or_truncate(
        token_ids,
        max_length=max_length,
        pad_token_id=vocab[PAD_TOKEN],
    )

    input_tensor = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(input_tensor)
        probabilities = torch.softmax(logits, dim=1)
        predicted_class = probabilities.argmax(dim=1).item()

    label_names = {0: "Negative", 1: "Positive"}
    return label_names[predicted_class], probabilities[0, predicted_class].item()

In [ ]:
example_reviews = [
    "This movie was absolutely fantastic! I loved every minute of it.",
    "This movie was terrible. The acting was awful and the story was boring.",
    "The movie was okay, but nothing special.",
]

for review in example_reviews:
    sentiment, score = predict_sentiment(review, model, vocab, device)
    print("Review:", review)
    print("Predicted sentiment:", sentiment)
    print(f"Softmax score: {score:.2%}")
    print("-" * 70)

## 12. Save the model and vocabulary

The inference script expects the model weights and vocabulary at these paths:

- `models/sentiment_lstm.pth`
- `models/vocab.json`

Run this section only after training a model you intend to keep. It overwrites those two files in the project directory, so make a backup first if you need to preserve the existing trained model.

In [ ]:
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "sentiment_lstm.pth"
vocab_path = models_dir / "vocab.json"

torch.save(model.state_dict(), model_path)
with open(vocab_path, "w", encoding="utf-8") as file:
    json.dump(vocab, file, ensure_ascii=False)

print("Saved model:", model_path)
print(f"Model size: {model_path.stat().st_size / (1024 ** 2):.2f} MB")
print("Saved vocabulary:", vocab_path)
print("Vocabulary size:", len(vocab))

## 13. Key takeaways

- Text is cleaned and tokenized before it is converted into integer IDs.
- The vocabulary is built from the training split only; `<PAD>` and `<UNK>` handle padding and unseen words.
- Embedding layers map token IDs to dense vectors.
- The LSTM summarizes the sequence, and a linear layer produces two class logits.
- `CrossEntropyLoss` is used for training; softmax is applied when displaying prediction scores.
- Test metrics estimate performance on held-out data, while individual predictions can still be wrong.

For command-line inference after saving the files, run this from the repository root:

```bash
python -m src.predict
```